# Module 2: Fixed Effects Diagnostics

## Learning Objectives

By completing this notebook, you will:
1. Test for serial correlation in panel data errors
2. Apply clustered standard errors correctly
3. Detect and handle heteroskedasticity
4. Perform specification tests for fixed effects
5. Conduct robustness checks

## Prerequisites

- Module 2.1: FE Implementation (completed)
- Understanding of OLS diagnostics
- Familiarity with hypothesis testing

## Estimated Time

75-90 minutes

---

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from linearmodels.panel import PanelOLS

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

In [ ]:
section_divider("1. Generate Panel Data with Violations")

In [ ]:
learning_objectives(["Test for serial correlation in panel data errors", "Apply clustered standard errors correctly", "Detect and handle heteroskedasticity", "Perform specification tests for fixed effects", "Conduct robustness checks"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Generate Panel Data with Violations

We'll create datasets that violate different assumptions to understand diagnostics.

In [ ]:
def generate_panel_data(n_entities=100, n_periods=10, 
                        serial_corr=0.0, heterosked=False):
    """
    Generate panel data with optional violations.
    
    Parameters
    ----------
    n_entities : int
        Number of entities
    n_periods : int
        Number of time periods
    serial_corr : float
        Serial correlation coefficient (0 = no correlation)
    heterosked : bool
        If True, variance depends on X
    
    Returns
    -------
    pd.DataFrame
    """
    # Panel structure
    entities = np.arange(1, n_entities + 1)
    periods = np.arange(1, n_periods + 1)
    
    panel_index = pd.MultiIndex.from_product(
        [entities, periods],
        names=['entity_id', 'time']
    )
    
    # Entity fixed effects
    alpha_i = np.random.randn(n_entities) * 20 + 50
    
    # Generate X (correlated with alpha_i)
    X = []
    for i in range(n_entities):
        base_x = 10 + 0.2 * alpha_i[i] + np.random.randn() * 3
        X_i = [base_x]
        for t in range(1, n_periods):
            X_i.append(0.6 * X_i[-1] + 0.4 * base_x + np.random.randn() * 2)
        X.extend(X_i)
    
    X = np.array(X)
    
    # Generate errors with optional serial correlation
    errors = []
    for i in range(n_entities):
        if heterosked:
            # Variance proportional to X
            sigma_i = 2 + 0.3 * abs(X[i*n_periods:(i+1)*n_periods])
        else:
            sigma_i = np.ones(n_periods) * 5
        
        # Generate errors with serial correlation
        e_i = [np.random.randn() * sigma_i[0]]
        for t in range(1, n_periods):
            e_i.append(serial_corr * e_i[-1] + 
                      np.sqrt(1 - serial_corr**2) * np.random.randn() * sigma_i[t])
        errors.extend(e_i)
    
    errors = np.array(errors)
    
    # Generate y
    alpha_repeated = np.repeat(alpha_i, n_periods)
    y = alpha_repeated + 1.5 * X + errors
    
    # Create DataFrame
    df = pd.DataFrame({
        'y': y,
        'x': X
    }, index=panel_index).reset_index()
    
    return df

# Generate clean data
df_clean = generate_panel_data(serial_corr=0.0, heterosked=False)

# Generate data with serial correlation
df_serial = generate_panel_data(serial_corr=0.5, heterosked=False)

# Generate data with heteroskedasticity
df_hetero = generate_panel_data(serial_corr=0.0, heterosked=True)

print("Generated three datasets:")
print("  1. Clean (no violations)")
print("  2. Serial correlation (rho = 0.5)")
print("  3. Heteroskedasticity")
print(f"\nEach has {len(df_clean)} observations")

In [ ]:
section_divider("2. Standard Errors: Why Clustering Matters")

## 2. Standard Errors: Why Clustering Matters

### The Problem

Panel data typically violates the independence assumption:
- Errors **within** an entity are correlated over time
- Standard OLS standard errors are **too small** (overconfident)

### The Solution

**Clustered standard errors** account for within-entity correlation.

### Demonstration

In [ ]:
# Estimate FE on serially correlated data
df_serial_panel = df_serial.set_index(['entity_id', 'time'])

# Model
model_fe = PanelOLS(
    dependent=df_serial_panel['y'],
    exog=df_serial_panel[['x']],
    entity_effects=True
)

# Estimate with different SE types
results_default = model_fe.fit()  # Default (homoskedastic)
results_robust = model_fe.fit(cov_type='robust')  # Heteroskedastic-robust
results_clustered = model_fe.fit(cov_type='clustered', cluster_entity=True)  # Clustered

# Compare standard errors
print("Standard Errors Comparison (Data with Serial Correlation)")
print("=" * 70)
print(f"{'SE Type':<25} {'Std Error':>12} {'t-stat':>10} {'95% CI Width':>15}")
print("=" * 70)

se_default = results_default.std_errors['x']
se_robust = results_robust.std_errors['x']
se_clustered = results_clustered.std_errors['x']

beta = results_default.params['x']

print(f"{'Default (homoskedastic)':<25} {se_default:>12.4f} {beta/se_default:>10.2f} {1.96*2*se_default:>15.4f}")
print(f"{'Robust':<25} {se_robust:>12.4f} {beta/se_robust:>10.2f} {1.96*2*se_robust:>15.4f}")
print(f"{'Clustered (by entity)':<25} {se_clustered:>12.4f} {beta/se_clustered:>10.2f} {1.96*2*se_clustered:>15.4f}")

print("\n" + "=" * 70)
print(f"Clustered SE is {se_clustered/se_default:.2f}x larger than default")
print("\n⚠️  With serial correlation, default SEs are too small!")
print("   Always use clustered SEs for panel data.")

### Visualization: Impact on Confidence Intervals

In [ ]:
# Create confidence intervals
ci_types = ['Default', 'Robust', 'Clustered']
ses = [se_default, se_robust, se_clustered]
ci_lower = [beta - 1.96*se for se in ses]
ci_upper = [beta + 1.96*se for se in ses]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

y_positions = np.arange(len(ci_types))

# Plot confidence intervals
for i, (lower, upper) in enumerate(zip(ci_lower, ci_upper)):
    ax.plot([lower, upper], [i, i], 'o-', linewidth=2, markersize=8)
    ax.plot([beta], [i], 'ro', markersize=10)

# True value line
ax.axvline(x=1.5, color='green', linestyle='--', linewidth=2, 
           label='True value (1.5)', alpha=0.7)

ax.set_yticks(y_positions)
ax.set_yticklabels(ci_types)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('95% Confidence Intervals: Impact of SE Choice', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("Note: Clustered CIs are wider (more conservative)")
print("      This correctly reflects uncertainty from serial correlation")

In [ ]:
section_divider("3. Testing for Serial Correlation")

## 3. Testing for Serial Correlation

### Wooldridge Test for Serial Correlation

Tests $H_0$: No first-order serial correlation in FE residuals.

**Procedure:**
1. Estimate FE model, get residuals $\hat{\epsilon}_{it}$
2. For each entity, regress $\hat{\epsilon}_{it}$ on $\hat{\epsilon}_{i,t-1}$
3. Test if average correlation is zero

In [ ]:
def wooldridge_test_serial_correlation(df, entity_col, time_col, 
                                        y_col, X_cols):
    """
    Wooldridge test for serial correlation in FE residuals.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    y_col : str
    X_cols : list
    
    Returns
    -------
    dict
        Test statistic and p-value
    """
    # Set panel index
    df_panel = df.set_index([entity_col, time_col])
    
    # Estimate FE
    model = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[X_cols],
        entity_effects=True
    )
    results = model.fit()
    
    # Get residuals
    residuals = results.resids
    df_resid = df.copy()
    df_resid['resid'] = residuals.values
    
    # Create lagged residuals
    df_resid = df_resid.sort_values([entity_col, time_col])
    df_resid['resid_lag'] = df_resid.groupby(entity_col)['resid'].shift(1)
    
    # Drop missing
    df_test = df_resid.dropna(subset=['resid_lag'])
    
    # Regress resid on resid_lag (pooled)
    X_test = sm.add_constant(df_test['resid_lag'])
    y_test = df_test['resid']
    
    test_model = sm.OLS(y_test, X_test)
    test_results = test_model.fit()
    
    # Test statistic (t-test on lag coefficient)
    rho = test_results.params['resid_lag']
    t_stat = test_results.tvalues['resid_lag']
    p_value = test_results.pvalues['resid_lag']
    
    return {
        'rho': rho,
        't_statistic': t_stat,
        'p_value': p_value
    }

# Test on clean data
print("Serial Correlation Tests")
print("=" * 70)

test_clean = wooldridge_test_serial_correlation(
    df_clean, 'entity_id', 'time', 'y', ['x']
)
print("\nClean data (no serial correlation):")
print(f"  Correlation coefficient: {test_clean['rho']:.4f}")
print(f"  t-statistic: {test_clean['t_statistic']:.3f}")
print(f"  p-value: {test_clean['p_value']:.4f}")

if test_clean['p_value'] > 0.05:
    print("  ✅ Cannot reject H0: No serial correlation")
else:
    print("  ⚠️  Reject H0: Serial correlation detected")

# Test on serially correlated data
test_serial = wooldridge_test_serial_correlation(
    df_serial, 'entity_id', 'time', 'y', ['x']
)
print("\nData with serial correlation (rho = 0.5):")
print(f"  Correlation coefficient: {test_serial['rho']:.4f}")
print(f"  t-statistic: {test_serial['t_statistic']:.3f}")
print(f"  p-value: {test_serial['p_value']:.4e}")

if test_serial['p_value'] > 0.05:
    print("  ✅ Cannot reject H0: No serial correlation")
else:
    print("  ⚠️  Reject H0: Serial correlation detected")

In [ ]:
section_divider("4. Testing for Heteroskedasticity")

## 4. Testing for Heteroskedasticity

### Modified Wald Test

Tests whether error variance differs across entities.

In [ ]:
def test_heteroskedasticity_panel(df, entity_col, time_col, y_col, X_cols):
    """
    Test for heteroskedasticity in panel data.
    
    Uses variance of residuals across entities.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    y_col : str
    X_cols : list
    
    Returns
    -------
    dict
        Test results
    """
    # Estimate FE
    df_panel = df.set_index([entity_col, time_col])
    model = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[X_cols],
        entity_effects=True
    )
    results = model.fit()
    
    # Get residuals
    residuals = results.resids
    df_resid = df.copy()
    df_resid['resid'] = residuals.values
    
    # Compute variance by entity
    entity_vars = df_resid.groupby(entity_col)['resid'].var()
    
    # Test if variances are equal (using coefficient of variation)
    mean_var = entity_vars.mean()
    std_var = entity_vars.std()
    cv = std_var / mean_var  # Coefficient of variation
    
    # Rough test: CV > 0.5 suggests heteroskedasticity
    has_hetero = cv > 0.5
    
    return {
        'mean_variance': mean_var,
        'std_variance': std_var,
        'cv': cv,
        'likely_heteroskedastic': has_hetero,
        'entity_variances': entity_vars
    }

# Test on clean data
print("Heteroskedasticity Tests")
print("=" * 70)

hetero_clean = test_heteroskedasticity_panel(
    df_clean, 'entity_id', 'time', 'y', ['x']
)
print("\nClean data (homoskedastic):")
print(f"  Mean variance: {hetero_clean['mean_variance']:.2f}")
print(f"  Std of variance: {hetero_clean['std_variance']:.2f}")
print(f"  Coefficient of variation: {hetero_clean['cv']:.3f}")

if hetero_clean['likely_heteroskedastic']:
    print("  ⚠️  Heteroskedasticity likely present")
else:
    print("  ✅ Homoskedasticity is reasonable")

# Test on heteroskedastic data
hetero_test = test_heteroskedasticity_panel(
    df_hetero, 'entity_id', 'time', 'y', ['x']
)
print("\nData with heteroskedasticity:")
print(f"  Mean variance: {hetero_test['mean_variance']:.2f}")
print(f"  Std of variance: {hetero_test['std_variance']:.2f}")
print(f"  Coefficient of variation: {hetero_test['cv']:.3f}")

if hetero_test['likely_heteroskedastic']:
    print("  ⚠️  Heteroskedasticity likely present")
else:
    print("  ✅ Homoskedasticity is reasonable")

### Visualize Heteroskedasticity

In [ ]:
# Estimate models
df_hetero_panel = df_hetero.set_index(['entity_id', 'time'])
model_hetero = PanelOLS(
    dependent=df_hetero_panel['y'],
    exog=df_hetero_panel[['x']],
    entity_effects=True
)
results_hetero = model_hetero.fit()

# Get residuals
df_hetero_resid = df_hetero.copy()
df_hetero_resid['resid'] = results_hetero.resids.values

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Residuals vs fitted
fitted = results_hetero.fitted_values.values
df_hetero_resid['fitted'] = fitted

axes[0].scatter(df_hetero_resid['fitted'], df_hetero_resid['resid'], 
                alpha=0.3, s=20)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Fitted Values', fontsize=11)
axes[0].set_ylabel('Residuals', fontsize=11)
axes[0].set_title('Residuals vs Fitted Values', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Panel 2: Distribution of entity variances
entity_vars = hetero_test['entity_variances']
axes[1].hist(entity_vars, bins=20, alpha=0.7, edgecolor='black')
axes[1].axvline(x=entity_vars.mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {entity_vars.mean():.1f}')
axes[1].set_xlabel('Residual Variance', fontsize=11)
axes[1].set_ylabel('Number of Entities', fontsize=11)
axes[1].set_title('Distribution of Entity-Level Variances', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Note: Funnel shape in left panel indicates heteroskedasticity")
print("      Wide spread in right panel confirms variance differs by entity")

In [ ]:
section_divider("5. Practical Recommendations")

## 5. Practical Recommendations

### Decision Tree for Standard Errors

```
Do you have panel data?
    |
    |-- Yes --> Use clustered SEs (cluster by entity)
    |           This handles both serial correlation AND heteroskedasticity
    |
    |-- No --> Do you have heteroskedasticity?
                |
                |-- Yes --> Use robust (HC) SEs
                |-- No --> Default SEs are fine
```

### Best Practice

**For panel data, ALWAYS use clustered standard errors.**

They are robust to:
- Serial correlation
- Heteroskedasticity
- Any within-entity correlation structure

---

## Exercises

### Exercise 1: Compute Robust F-Test

**Task:** F-tests for coefficient restrictions need to account for clustered errors. Implement a robust F-test.

**Test:** $H_0: \beta_1 = 0$ (standard t-test, but demonstrate principle)

**Hints:**
- Compare models with/without restriction
- Use clustered SEs for both
- F-stat depends on SE choice

In [ ]:
# YOUR CODE HERE
def robust_f_test(df, entity_col, time_col, y_col, X_cols, 
                  test_vars, cov_type='clustered'):
    """
    Perform F-test with robust standard errors.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    y_col : str
    X_cols : list
        All predictors
    test_vars : list
        Variables to test (subset of X_cols)
    cov_type : str
        Type of SEs ('clustered', 'robust', 'unadjusted')
    
    Returns
    -------
    dict
        F-stat and p-value
    """
    # TODO: Implement robust F-test
    pass

In [ ]:
# SOLUTION (hidden)
def robust_f_test_solution(df, entity_col, time_col, y_col, X_cols, 
                           test_vars, cov_type='clustered'):
    """Perform F-test with robust standard errors."""
    df_panel = df.set_index([entity_col, time_col])
    
    # Full model
    model_full = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[X_cols],
        entity_effects=True
    )
    
    if cov_type == 'clustered':
        results_full = model_full.fit(cov_type='clustered', cluster_entity=True)
    else:
        results_full = model_full.fit(cov_type=cov_type)
    
    # Restricted model (exclude test_vars)
    X_restricted = [x for x in X_cols if x not in test_vars]
    
    if len(X_restricted) > 0:
        model_restricted = PanelOLS(
            dependent=df_panel[y_col],
            exog=df_panel[X_restricted],
            entity_effects=True
        )
        
        if cov_type == 'clustered':
            results_restricted = model_restricted.fit(cov_type='clustered', 
                                                      cluster_entity=True)
        else:
            results_restricted = model_restricted.fit(cov_type=cov_type)
        
        ssr_full = results_full.resid_ss
        ssr_restricted = results_restricted.resid_ss
    else:
        # Restricted model is just entity FE (no X)
        ssr_restricted = ((df_panel[y_col] - df_panel[y_col].mean())**2).sum()
        ssr_full = results_full.resid_ss
    
    # F-statistic
    q = len(test_vars)
    n = len(df)
    k = len(X_cols)
    
    f_stat = ((ssr_restricted - ssr_full) / q) / (ssr_full / (n - k))
    p_value = 1 - stats.f.cdf(f_stat, q, n - k)
    
    return {
        'f_statistic': f_stat,
        'df1': q,
        'df2': n - k,
        'p_value': p_value
    }

In [ ]:
# Test your function
def test_exercise_1():
    """Test robust F-test implementation."""
    f_test_result = robust_f_test(
        df_serial, 'entity_id', 'time', 'y', ['x'], ['x'], 
        cov_type='clustered'
    )
    
    assert f_test_result is not None, "Function returns None - did you implement it?"
    assert 'f_statistic' in f_test_result, "Missing f_statistic"
    assert 'p_value' in f_test_result, "Missing p_value"
    
    print("✅ Correct! Robust F-test implemented.")
    print("\nTest: H0: coefficient on x = 0")
    print("=" * 50)
    print(f"F-statistic: {f_test_result['f_statistic']:.2f}")
    print(f"Degrees of freedom: ({f_test_result['df1']}, {f_test_result['df2']})")
    print(f"P-value: {f_test_result['p_value']:.4e}")
    
    if f_test_result['p_value'] < 0.01:
        print("\n✅ Reject H0: x has significant effect")
    else:
        print("\n⚠️  Cannot reject H0")
    
    return True

# Uncomment to test
# test_exercise_1()

### Exercise 2: Residual Diagnostics Plot

**Task:** Create a comprehensive residual diagnostics plot for FE models.

**Include:**
1. Residuals vs fitted values
2. Q-Q plot for normality
3. Residual histogram
4. Residuals over time (sample entities)

**Hints:**
- Use `scipy.stats.probplot` for Q-Q plot
- Create 2x2 subplot grid

In [ ]:
# YOUR CODE HERE
def plot_fe_diagnostics(df, entity_col, time_col, y_col, X_cols, 
                        n_sample_entities=4):
    """
    Create diagnostic plots for FE model.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    y_col : str
    X_cols : list
    n_sample_entities : int
        Number of entities to show in time plot
    """
    # TODO: Create 2x2 diagnostic plot
    pass

In [ ]:
# SOLUTION (hidden)
def plot_fe_diagnostics_solution(df, entity_col, time_col, y_col, X_cols, 
                                  n_sample_entities=4):
    """Create diagnostic plots for FE model."""
    # Estimate model
    df_panel = df.set_index([entity_col, time_col])
    model = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[X_cols],
        entity_effects=True
    )
    results = model.fit()
    
    # Get residuals and fitted values
    df_diag = df.copy()
    df_diag['resid'] = results.resids.values
    df_diag['fitted'] = results.fitted_values.values
    
    # Create plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Residuals vs Fitted
    axes[0, 0].scatter(df_diag['fitted'], df_diag['resid'], alpha=0.3, s=20)
    axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
    axes[0, 0].set_xlabel('Fitted Values', fontsize=11)
    axes[0, 0].set_ylabel('Residuals', fontsize=11)
    axes[0, 0].set_title('Residuals vs Fitted', fontsize=12)
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Q-Q Plot
    stats.probplot(df_diag['resid'], dist="norm", plot=axes[0, 1])
    axes[0, 1].set_title('Q-Q Plot', fontsize=12)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Histogram
    axes[1, 0].hist(df_diag['resid'], bins=30, alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 0].set_xlabel('Residuals', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title('Residual Distribution', fontsize=12)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # 4. Residuals over time
    sample_entities = np.random.choice(
        df[entity_col].unique(), n_sample_entities, replace=False
    )
    for entity in sample_entities:
        entity_data = df_diag[df_diag[entity_col] == entity]
        axes[1, 1].plot(entity_data[time_col], entity_data['resid'], 
                       marker='o', alpha=0.7, label=f'Entity {entity}')
    
    axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
    axes[1, 1].set_xlabel('Time', fontsize=11)
    axes[1, 1].set_ylabel('Residuals', fontsize=11)
    axes[1, 1].set_title('Residuals Over Time', fontsize=12)
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Test your function
def test_exercise_2():
    """Test diagnostic plots."""
    print("Creating diagnostic plots for data with serial correlation...")
    plot_fe_diagnostics(df_serial, 'entity_id', 'time', 'y', ['x'])
    
    print("\n✅ Diagnostic plots created!")
    print("\nInterpretation:")
    print("  - Top-left: Look for patterns (funnel = heterosked)")
    print("  - Top-right: Deviations from line = non-normality")
    print("  - Bottom-left: Should be roughly symmetric")
    print("  - Bottom-right: Look for time patterns (serial correlation)")
    
    return True

# Uncomment to test
# test_exercise_2()

### Exercise 3: Power Analysis for Panel Data

**Task:** Investigate how serial correlation affects statistical power.

**Approach:**
1. Simulate data with varying serial correlation (rho = 0, 0.3, 0.6)
2. Estimate FE with clustered SEs
3. Calculate rejection rate of H0: beta = 0 at alpha = 0.05
4. Compare power across correlation levels

**Hints:**
- Run 100-200 simulations per rho
- Count how often |t-stat| > 1.96

In [ ]:
# YOUR CODE HERE
def power_analysis_serial_correlation(rho_values=[0, 0.3, 0.6], 
                                       n_sims=100, alpha=0.05):
    """
    Analyze power under serial correlation.
    
    Parameters
    ----------
    rho_values : list
        Serial correlation coefficients to test
    n_sims : int
        Number of simulations per rho
    alpha : float
        Significance level
    
    Returns
    -------
    dict
        Power estimates for each rho
    """
    # TODO: Implement power analysis
    pass

In [ ]:
# SOLUTION (hidden)
def power_analysis_serial_correlation_solution(rho_values=[0, 0.3, 0.6], 
                                                n_sims=100, alpha=0.05):
    """Analyze power under serial correlation."""
    results = {}
    
    for rho in rho_values:
        rejections = 0
        
        for sim in range(n_sims):
            # Generate data
            df_sim = generate_panel_data(
                n_entities=50, n_periods=8, 
                serial_corr=rho, heterosked=False
            )
            
            # Estimate with clustered SEs
            df_sim_panel = df_sim.set_index(['entity_id', 'time'])
            model = PanelOLS(
                dependent=df_sim_panel['y'],
                exog=df_sim_panel[['x']],
                entity_effects=True
            )
            results_sim = model.fit(cov_type='clustered', cluster_entity=True)
            
            # Test H0: beta = 0
            t_stat = results_sim.tstats['x']
            if abs(t_stat) > stats.norm.ppf(1 - alpha/2):
                rejections += 1
        
        power = rejections / n_sims
        results[rho] = power
    
    return results

In [ ]:
# Test your function (WARNING: This may take 1-2 minutes)
def test_exercise_3():
    """Test power analysis."""
    print("Running power analysis (this may take a minute)...")
    
    power_results = power_analysis_serial_correlation(
        rho_values=[0, 0.3, 0.6],
        n_sims=50,  # Reduced for speed
        alpha=0.05
    )
    
    assert power_results is not None, "Function returns None - did you implement it?"
    assert len(power_results) == 3, "Should have 3 power estimates"
    
    print("\n✅ Correct! Power analysis complete.")
    print("\nPower (rejection rate) under different serial correlations:")
    print("=" * 50)
    for rho, power in power_results.items():
        print(f"  rho = {rho:.1f}: Power = {power:.3f}")
    
    print("\nNote: Power decreases with serial correlation")
    print("      because clustered SEs are larger (more conservative)")
    
    return True

# Uncomment to test (takes ~1 minute)
# test_exercise_3()

---

## Summary

### Key Takeaways

1. **Clustered Standard Errors Are Essential:** Panel data almost always has within-entity correlation; default SEs are too small

2. **Serial Correlation:** Wooldridge test detects first-order correlation in residuals

3. **Heteroskedasticity:** Variance often differs across entities; test using entity-level variance

4. **Clustered SEs Handle Both:** One solution for serial correlation AND heteroskedasticity

5. **Diagnostic Plots:** Essential for validating model assumptions

### Practical Workflow

1. Estimate FE with clustered SEs
2. Test for serial correlation (Wooldridge test)
3. Examine residual plots
4. If violations severe, consider:
   - First differences (serial correlation)
   - Robust/clustered SEs (heteroskedasticity)
   - Different model specification

### What's Next

In the next notebook, we'll extend to **two-way fixed effects** (entity + time), which controls for both entity-specific and time-specific unobservables.

---

**Next:** `03_twfe_practice.ipynb` - Two-Way Fixed Effects

In [ ]:
key_takeaways(["Next:"])